 Faisa Hassan Sheikahmed - SHE23601467
 
 Last updated - 16.03.25

# AI coursework 2 - Sustainable transportation solutions

In [ ]:
!pip install plotly

In [ ]:
pip install requests

In [ ]:
import plotly.graph_objects as go

In [ ]:
# Task 1 - Define the Map
graph = {
    "Manchester": [("Liverpool", 40), ("York", 60), ("Carlisle", 120), ("Edinburgh", 220)],
    "Liverpool": [("Manchester", 40), ("Holyhead", 90)],
    "Holyhead": [("Liverpool", 90)],
    "York": [("Manchester", 60), ("Newcastle", 80)],
    "Newcastle": [("York", 80), ("Edinburgh", 110)],
    "Carlisle": [("Manchester", 120), ("Glasgow", 100)],
    "Edinburgh": [("Manchester", 220), ("Newcastle", 110), ("Glasgow", 40)],
    "Glasgow": [("Carlisle", 100), ("Edinburgh", 40), ("Oban", 90), ("Aberdeen", 140), ("Inverness", 170)],
    "Oban": [("Glasgow", 90), ("Inverness", 110)],
    "Aberdeen": [("Glasgow", 140), ("Inverness", 110)],
    "Inverness": [("Glasgow", 170), ("Oban", 110), ("Aberdeen", 110)]
}

# Task 1 - Ask user to select start and end cities
def get_user_input():
    print("Eco-Friendly Delivery Made Smart!** 🌍📦 \nOur AI finds the greenest, most efficient routes, reducing emissions and costs for smarter deliveries. 🚚💨🌱\n")
    print("Available cities:\n")
    for city in graph.keys():
        print(f"- {city}")

    start_city = input("\nEnter the starting city: ").capitalize()
    end_city = input("Enter the ending city: ").capitalize()

    # Validate input
    while start_city not in graph or end_city not in graph:
        print("Invalid cities entered. Please choose from the available cities.")
        start_city = input("Enter the starting city: ").capitalize()
        end_city = input("Enter the ending city: ").capitalize()

    return start_city, end_city

# Sample usage
#start_city, end_city = get_user_input()
#print(f"Starting from {start_city} to {end_city}.\n")


In [ ]:
# Sample coordinates for each city
city_coords = {
    "Manchester": (53.4808, -2.2426),
    "Liverpool": (53.4084, -2.9916),
    "Holyhead": (53.3040, -4.6307),
    "York": (53.9590, -1.0815),
    "Newcastle": (54.9783, -1.6178),
    "Carlisle": (54.8919, -2.9320),
    "Edinburgh": (55.9533, -3.1883),
    "Glasgow": (55.8642, -4.2518),
    "Oban": (56.4111, -5.4729),
    "Aberdeen": (57.1497, -2.0943),
    "Inverness": (57.4778, -4.2247)
}

# Define function to create the map
def create_map(path=[]):
    # Create a Plotly figure
    fig = go.Figure()

    # Add city markers
    for city, coord in city_coords.items():
        fig.add_trace(go.Scattergeo(
            locationmode="country names",  # Using country names for better accuracy
            lon=[coord[1]],
            lat=[coord[0]],
            mode="markers+text",
            text=city,
            marker=dict(size=10, color="blue"),
            textposition="top right"
        ))

    # Add lines for the best route (if provided)
    if path:
        for i in range(len(path) - 1):
            start_city = path[i]
            end_city = path[i + 1]
            start_lat, start_lon = city_coords[start_city]
            end_lat, end_lon = city_coords[end_city]
            fig.add_trace(go.Scattergeo(
                locationmode="country names",
                lon=[start_lon, end_lon],
                lat=[start_lat, end_lat],
                mode="lines",
                line=dict(width=4, color="red"),  # Highlight the path in red
                opacity=0.7
            ))

    # Set layout for the map (zoom in on England)
    fig.update_layout(
        title="Route map of the shortest path)",
        geo=dict(
            scope="europe",  # Focus on Europe
            center=dict(lat=52.5, lon=-1.5),  # Center on England (around the middle of England)
            projection_type="mercator",  # Mercator projection
            projection_scale=8,  # Adjust the zoom level to focus on England
            showland=True,
            landcolor="lightblue",
            countrycolor="black",
            showlakes=True,  # Show lakes
            lakecolor="blue"
        ),
        height=800,  # Set height of the map (adjust as needed)
        width=1000   # Set width of the map (adjust as needed)
    )

    # Show the map
    fig.show()

In [ ]:
# Accurate Carbon emission factor (kg CO₂ per km) for a petrol car
CARBON_EMISSION_FACTOR = 0.196974607  

# Function to calculate carbon emissions for a given distance
def calculate_carbon_emission(distance_miles):
    """Calculate carbon emissions based on the distance in miles."""
    
    # Convert miles to kilometers
    distance_km = distance_miles * 1.60934  # 1 mile = 1.60934 kilometers
    
    # Calculate carbon emissions
    carbon_emission = distance_km * CARBON_EMISSION_FACTOR
    return carbon_emission

In [ ]:
# Depth-First Search (DFS) Algorithm to find the path and calculate carbon emissions
def dfs(graph, start, end, path=None, distance=0):
    if path is None:
        path = []

    # Add current city to the path
    path.append(start)

    # If we have reached the end city, return the path and distance
    if start == end:
        # Calculate carbon emissions based on the distance
        carbon_emission = calculate_carbon_emission(distance)
        return path, distance, carbon_emission

    # Explore neighbors (cities) recursively
    for neighbor, dist in graph[start]:
        if neighbor not in path:  # Avoid cycles by not revisiting cities
            new_path, new_distance, _ = dfs(graph, neighbor, end, path.copy(), distance + dist)  # Recursively explore
            if new_path:  # If a valid path is found
                # Calculate carbon emissions for the found path
                carbon_emission = calculate_carbon_emission(new_distance)
                return new_path, new_distance, carbon_emission

    return None, None, None  # If no path is found


In [ ]:
# BFS Implementation to find the path, distance, and carbon emissions
from collections import deque

def bfs(graph, start, end):
    # Initialize the queue and visited set
    queue = deque([(start, [start], 0)])  # (current city, path, total distance)
    visited = set()  # To track visited cities

    # Start BFS
    while queue:
        current_city, path, total_distance = queue.popleft()

        # If we have reached the destination city, return the path, total distance, and carbon emissions
        if current_city == end:
            # Calculate carbon emissions based on the distance
            carbon_emission = calculate_carbon_emission(total_distance)
            return path, total_distance, carbon_emission

        # Explore neighbors (go to the next level)
        for neighbor, dist in graph[current_city]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor], total_distance + dist))

    return None, None, None  # If no path is found

In [ ]:
# A* algorithm
import heapq

def heuristic(city, end_city, city_coords):
    """Returns the straight-line (Euclidean) distance as the heuristic."""
    x1, y1 = city_coords[city]
    x2, y2 = city_coords[end_city]
    return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5  # Calculate Euclidean distance

def a_star(graph, city_coords, start_city, end_city):
    """Implements A* algorithm to find the shortest path between cities."""
    open_set = []  # Priority queue to explore cities
    heapq.heappush(open_set, (0, start_city))  # Add start city to open set with cost 0
    came_from = {}  # Dictionary to track the path
    g_score = {city: float('inf') for city in graph}  # Initialize g-scores
    g_score[start_city] = 0  # Start city has zero g-score
    f_score = {city: float('inf') for city in graph}  # Initialize f-scores
    f_score[start_city] = heuristic(start_city, end_city, city_coords)  # Start city's heuristic
    
    while open_set:
        _, current = heapq.heappop(open_set)  # Get city with lowest f-score
        
        if current == end_city:  # If destination is reached
            path = []  # Initialize path
            distance = g_score[current]  # Total distance to destination
            while current in came_from:  # Reconstruct the path
                path.append(current)
                current = came_from[current]
            path.append(start_city)  # Add start city to path
            path = path[::-1]  # Reverse path to start -> end
            
            # Calculate carbon emissions based on distance
            carbon_emission = calculate_carbon_emission(distance)
            
            return path, distance, carbon_emission  # Return path, distance, and carbon emission
        
        for neighbor, cost in graph[current]:  # Explore neighbors
            tentative_g_score = g_score[current] + cost  # Calculate tentative g-score
            if tentative_g_score < g_score[neighbor]:  # If better path found
                came_from[neighbor] = current  # Update path
                g_score[neighbor] = tentative_g_score  # Update g-score
                f_score[neighbor] = tentative_g_score + heuristic(neighbor, end_city, city_coords)  # Update f-score
                heapq.heappush(open_set, (f_score[neighbor], neighbor))  # Add neighbor to open set
    
    return None, float('inf'), None  # No path found


In [ ]:
# My 4th algorithm -> Dijkstra algorithm
"""
    I chose Dijkstra's algorithm because it efficiently finds the shortest path in graphs 
    with non-negative edge weights. It ensures the optimal route by exploring the nearest neighbors 
    first, making it ideal for applications like navigation or delivery route optimization.
"""

def dijkstra(graph, start, end):
    """Implements Dijkstra's algorithm to find the shortest path between cities."""
    priority_queue = [(0, start, [start])]  # (total_cost, current_city, path_so_far)
    distances = {city: float('inf') for city in graph}  # Initialize distances to infinity
    distances[start] = 0  # Start city has distance 0

    while priority_queue:
        current_distance, current_node, path = heapq.heappop(priority_queue)  # Get city with smallest distance

        if current_node == end:  # If destination reached
            carbon_emission = calculate_carbon_emission(current_distance)  # Calculate carbon emissions
            return path, current_distance, carbon_emission  # Return path, distance, and carbon emissions

        for neighbor, weight in graph[current_node]:  # Explore neighbors
            distance = current_distance + weight  # Calculate new distance to neighbor
            if distance < distances[neighbor]:  # If shorter path found
                distances[neighbor] = distance  # Update distance
                heapq.heappush(priority_queue, (distance, neighbor, path + [neighbor]))  # Add neighbor to queue

    return None, float("inf"), None  # No path found


In [ ]:
def print_result(algorithm_name, start_city, end_city, path, distance, carbon_emission):
    """
    Prints the result of the algorithm's execution, showing the route, total distance, 
    and carbon emissions.
    """
    # If no path was found, print an error message
    if path is None:
        print(f"❌ {algorithm_name} could not find a path from {start_city} to {end_city}.")
        return

    # Print the route found by the algorithm
    print(f"🚀 {algorithm_name} found this route:")

    # Loop through the path and print each segment
    for i in range(len(path) - 1):
        city1, city2 = path[i], path[i+1]
        
        # Get the distance between the two cities
        road_distance = next(weight for neighbor, weight in graph[city1] if neighbor == city2)
        
        # Print the road segment with the distance
        print(f"   📍 {city1} → {city2} = {road_distance} miles")

    # Print total distance and carbon emissions
    print(f"🏁 Total distance: {distance} miles")
    print(f"💨 Carbon Emissions: {carbon_emission:.2f} kg CO₂\n")


In [ ]:
# Main function
def main():
    start_city, end_city = get_user_input()

    # Run all algorithms
    dfs_path, dfs_distance, dfs_carbon = dfs(graph, start_city, end_city)
    bfs_path, bfs_distance, bfs_carbon = bfs(graph, start_city, end_city)
    a_star_path, a_star_distance, a_star_carbon = a_star(graph, city_coords, start_city, end_city)
    dijkstra_path, dijkstra_distance, dijkstra_carbon = dijkstra(graph, start_city, end_city)

    # Display results
    print_result("Depth-First Search (DFS)", start_city, end_city, dfs_path, dfs_distance, dfs_carbon)
    print_result("Breadth-First Search (BFS)", start_city, end_city, bfs_path, bfs_distance, bfs_carbon)
    print_result("A* Search", start_city, end_city, a_star_path, a_star_distance, a_star_carbon)
    print_result("Dijkstra’s Algorithm", start_city, end_city, dijkstra_path, dijkstra_distance, dijkstra_carbon)


     # Find the shortest path (based on the smallest distance)
    distances = {
        "Depth-First Search (DFS)": dfs_distance,
        "Breadth-First Search (BFS)": bfs_distance,
        "A* Search": a_star_distance,
        "Dijkstra’s Algorithm": dijkstra_distance
    }
    
    # Get the algorithm with the shortest distance
    shortest_algorithm = min(distances, key=distances.get)
    shortest_distance = distances[shortest_algorithm]
    
    # Select the path corresponding to the shortest distance
    if shortest_algorithm == "Depth-First Search (DFS)":
        shortest_path = dfs_path
    elif shortest_algorithm == "Breadth-First Search (BFS)":
        shortest_path = bfs_path
    elif shortest_algorithm == "A* Search":
        shortest_path = a_star_path
    else:
        shortest_path = dijkstra_path 

# Display the shortest path and algorithm name
    print(f"\nFor the shortest path from {start_city} to {end_city},\nWe can use the {shortest_algorithm} algorithm with a distance of {shortest_distance} miles.")
    
    # Display the map with the shortest path
    create_map(shortest_path)
    

if __name__ == "__main__":
    main()